Assignment 2 – Scraping, APIs y cruce por ubigeo    
    Parte 1------Temporada analizada: 1 de enero al 31 de mayo de 2022.

#1.Revisión

Antes de realizar el scraping, se revisa el archivo robots.txt de gob.pe para identificar las restricciones establecidas.

In [1]:
import requests

url_robots = "https://www.gob.pe/robots.txt"

headers = {
    "User-Agent": "Mozilla/5.0"
}

respuesta = requests.get(url_robots, headers=headers)

print("Código de estado:", respuesta.status_code)
print(respuesta.text)

Código de estado: 200
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



#Interpretación

El archivo robots.txt establece restricciones para el acceso automatizado a determinadas rutas de gob.pe, como /admin/ y algunas direcciones que utilizan el parámetro sheet.

La ruta /busquedas no aparece prohibida por estas reglas, por lo que se puede utilizar para realizar las consultas del trabajo; sin embargo, es importante incorporar pausas entre las solicitudes para evitar sobrecargar el servidor. El archivo también incluye indicaciones de Crawl-delay para determinados robots, lo que demuestra la importancia de controlar la frecuencia de acceso.

#2. Prueba de Selenium con un mes

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
url_enero = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas&"
    "institucion[]=pcm&"
    "term=estado%20de%20emergencia%20precipitaciones&"
    "sort_by=recent&"
    "desde=01-01-2022&"
    "hasta=31-01-2022"
)

print(url_enero)

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2022&hasta=31-01-2022


In [4]:
driver = webdriver.Chrome()

driver.get(url_enero)

wait = WebDriverWait(driver, 20)

articles = wait.until(
    EC.presence_of_all_elements_located((By.TAG_NAME, "article"))
)

print("Cantidad de artículos encontrados:", len(articles))

Cantidad de artículos encontrados: 8


#3. Número total de resultados

In [5]:
texto_pagina = driver.find_element(
    By.TAG_NAME, "body"
).text

lineas = texto_pagina.split("\n")

lineas_resultados = []

for linea in lineas:
    if "Resultados" in linea:
        lineas_resultados.append(linea)

print(lineas_resultados)

['8 Resultados']


In [6]:
texto_resultados = lineas_resultados[0]

partes = texto_resultados.split()

resultados_totales = int(partes[0])

print("Resultados totales:", resultados_totales)

Resultados totales: 8


#4. Extracción de información de resultados

In [7]:
datos_enero = []

for article in articles:
    
    enlaces = article.find_elements(By.TAG_NAME, "a")
    
    numero = enlaces[0].text.strip()
    enlace = enlaces[0].get_attribute("href")
    
    lineas = article.text.split("\n")
    
    fecha = None
    titulo = None
    
    for i in range(len(lineas)):
        
        if lineas[i].startswith("Publicado:"):
            fecha = lineas[i].replace("Publicado:", "").strip()
            
            if i + 1 < len(lineas):
                titulo = lineas[i + 1].strip()
    
    registro = {
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    }
    
    datos_enero.append(registro)

In [8]:
import pandas as pd

df_enero = pd.DataFrame(datos_enero)

df_enero

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 011-2022-PCM,30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 010-2022-PCM,29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 009-2022-PCM,28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 004-2022-PCM,25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 007-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 006-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 005-2022-PCM,16 de enero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Decreto Supremo N.° 002-2022-PCM,6 de enero de 2022,Decreto Supremo que modifica el numeral 8.1 de...,https://www.gob.pe/institucion/pcm/normas-lega...


#5. Extracción total

In [9]:
import time

meses = [
    ("enero", "01-01-2022", "31-01-2022"),
    ("febrero", "01-02-2022", "28-02-2022"),
    ("marzo", "01-03-2022", "31-03-2022"),
    ("abril", "01-04-2022", "30-04-2022"),
    ("mayo", "01-05-2022", "31-05-2022")
]

In [10]:
todos_datos = []
verificacion = []

for mes, desde, hasta in meses:

    url = (
        "https://www.gob.pe/busquedas?"
        "contenido[]=normas&"
        "institucion[]=pcm&"
        "term=estado%20de%20emergencia%20precipitaciones&"
        "sort_by=recent&"
        f"desde={desde}&"
        f"hasta={hasta}"
    )

    driver.get(url)

    articles = WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located(
            (By.TAG_NAME, "article")
        )
    )

    texto_pagina = driver.find_element(
        By.TAG_NAME, "body"
    ).text

    lineas_pagina = texto_pagina.split("\n")

    resultados_totales = 0

    for linea in lineas_pagina:
        if "Resultados" in linea:
            partes = linea.split()
            resultados_totales = int(partes[0])
            break

    datos_mes = []

    for article in articles:

        enlaces = article.find_elements(By.TAG_NAME, "a")

        numero = enlaces[0].text.strip()
        enlace = enlaces[0].get_attribute("href")

        lineas = article.text.split("\n")

        fecha = None
        titulo = None

        for i in range(len(lineas)):

            if lineas[i].startswith("Publicado:"):

                fecha = lineas[i].replace(
                    "Publicado:", ""
                ).strip()

                if i + 1 < len(lineas):
                    titulo = lineas[i + 1].strip()

        registro = {
            "mes": mes,
            "numero": numero,
            "fecha": fecha,
            "titulo": titulo,
            "enlace": enlace
        }

        datos_mes.append(registro)
        todos_datos.append(registro)

    resumen_mes = {
        "mes": mes,
        "resultados_totales": resultados_totales,
        "resultados_extraidos": len(datos_mes)
    }

    verificacion.append(resumen_mes)

    print(
        mes,
        "- Totales:", resultados_totales,
        "- Extraídos:", len(datos_mes)
    )

    time.sleep(2)

enero - Totales: 8 - Extraídos: 8
febrero - Totales: 5 - Extraídos: 5
marzo - Totales: 5 - Extraídos: 5
abril - Totales: 4 - Extraídos: 4
mayo - Totales: 7 - Extraídos: 7


In [11]:
df_decretos = pd.DataFrame(todos_datos)

print("Total de registros:", len(df_decretos))

df_decretos.head()

Total de registros: 29


,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 011-2022-PCM,30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 010-2022-PCM,29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 009-2022-PCM,28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 004-2022-PCM,25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 007-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


#6. Verificación de resultados por mes

In [12]:
df_verificacion = pd.DataFrame(verificacion)

df_verificacion

,mes,resultados_totales,resultados_extraidos
0,enero,8,8
1,febrero,5,5
2,marzo,5,5
3,abril,4,4
4,mayo,7,7


In [13]:
df_verificacion["coinciden"] = (
    df_verificacion["resultados_totales"]
    == df_verificacion["resultados_extraidos"]
)

display(df_verificacion)

print(
    "¿Coinciden todos los meses?",
    df_verificacion["coinciden"].all()
)

,mes,resultados_totales,resultados_extraidos,coinciden
0,enero,8,8,True
1,febrero,5,5,True
2,marzo,5,5,True
3,abril,4,4,True
4,mayo,7,7,True


¿Coinciden todos los meses? True


#7. Eliminación de normas repetidas

In [14]:
duplicados = df_decretos.duplicated(
    subset="enlace"
).sum()

print("Total de registros:", len(df_decretos))
print("Normas duplicadas:", duplicados)

Total de registros: 29
Normas duplicadas: 0


In [15]:
df_decretos = df_decretos.drop_duplicates(
    subset="enlace"
)

df_decretos = df_decretos.reset_index(
    drop=True
)

print(
    "Registros después de eliminar duplicados:",
    len(df_decretos)
)

Registros después de eliminar duplicados: 29


#8. Identificación de decretos

In [16]:
cantidad_inicial = len(df_decretos)

df_decretos = df_decretos[
    df_decretos["enlace"].str.contains(
        "/institucion/pcm/",
        regex=False,
        na=False
    )
].copy().reset_index(drop=True)

print("Normas de otras instituciones eliminadas:",
      cantidad_inicial - len(df_decretos))

print("Normas de la PCM:", len(df_decretos))

Normas de otras instituciones eliminadas: 0
Normas de la PCM: 29


#9. Recuperación de los títulos completos

In [17]:
import requests
import time
from bs4 import BeautifulSoup

titulos_completos = []

for enlace in df_decretos["enlace"]:

    respuesta = requests.get(
        enlace,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    respuesta.raise_for_status()

    sopa = BeautifulSoup(
        respuesta.text,
        "html.parser"
    )

    descripcion = sopa.find(
        "div",
        class_="description"
    )

    if descripcion:
        titulo = descripcion.get_text(" ", strip=True)
    else:
        titulo = ""

    titulos_completos.append(titulo)

    time.sleep(1)

df_decretos["titulo_completo"] = titulos_completos

print("Total de normas:", len(df_decretos))
print("Títulos recuperados:",
      (df_decretos["titulo_completo"] != "").sum())
print("Títulos pendientes:",
      (df_decretos["titulo_completo"] == "").sum())

display(
    df_decretos[
        ["numero", "titulo_completo"]
    ]
)

Total de normas: 29
Títulos recuperados: 29
Títulos pendientes: 0


,numero,titulo_completo
0,Decreto Supremo N.° 011-2022-PCM,Decreto Supremo que modifica el numeral 14.2 d...
1,Decreto Supremo N.° 010-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...
2,Decreto Supremo N.° 009-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...
3,Decreto Supremo N.° 004-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...
4,Decreto Supremo N.° 007-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...
5,Decreto Supremo N.° 006-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...
6,Decreto Supremo N.° 005-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...
7,Decreto Supremo N.° 002-2022-PCM,Decreto Supremo que modifica el numeral 8.1 de...
8,Decreto Supremo N.° 016-2022-PCM,Decreto Supremo que declara Estado de Emergenc...
9,Decreto Supremo N.° 015-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...


#10. Clasificación de las normas

In [18]:
import pandas as pd

def clasificar_norma(titulo):
    titulo = titulo.lower()

    es_lluvia = "lluvia" in titulo or "precipitaciones" in titulo

    if "prórroga" in titulo or "prorroga" in titulo:
        tipo = "prorroga"
    elif "declara" in titulo:
        tipo = "declara"
    else:
        tipo = "otro"

    if "peligro inminente" in titulo:
        motivo = "peligro inminente"
    elif "impacto de daños" in titulo:
        motivo = "impacto de daños"
    else:
        motivo = "otro"

    return es_lluvia, tipo, motivo


df_decretos[
    ["es_lluvia", "tipo", "motivo"]
] = df_decretos["titulo_completo"].apply(
    lambda titulo: pd.Series(clasificar_norma(titulo))
)

display(
    df_decretos[
        ["numero", "es_lluvia", "tipo", "motivo"]
    ]
)

,numero,es_lluvia,tipo,motivo
0,Decreto Supremo N.° 011-2022-PCM,False,declara,otro
1,Decreto Supremo N.° 010-2022-PCM,False,prorroga,otro
2,Decreto Supremo N.° 009-2022-PCM,False,prorroga,impacto de daños
3,Decreto Supremo N.° 004-2022-PCM,False,prorroga,otro
4,Decreto Supremo N.° 007-2022-PCM,False,prorroga,otro
5,Decreto Supremo N.° 006-2022-PCM,False,prorroga,otro
6,Decreto Supremo N.° 005-2022-PCM,False,declara,otro
7,Decreto Supremo N.° 002-2022-PCM,False,declara,otro
8,Decreto Supremo N.° 016-2022-PCM,False,declara,otro
9,Decreto Supremo N.° 015-2022-PCM,False,declara,otro


In [19]:
df_otros = df_decretos[
    (df_decretos["tipo"] == "otro") |
    (df_decretos["motivo"] == "otro")
]

print("Normas con alguna clasificación 'otro':", len(df_otros))

display(
    df_otros[
        ["numero", "titulo_completo", "tipo", "motivo"]
    ]
)

Normas con alguna clasificación 'otro': 22


,numero,titulo_completo,tipo,motivo
0,Decreto Supremo N.° 011-2022-PCM,Decreto Supremo que modifica el numeral 14.2 d...,declara,otro
1,Decreto Supremo N.° 010-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,prorroga,otro
3,Decreto Supremo N.° 004-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,prorroga,otro
4,Decreto Supremo N.° 007-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,prorroga,otro
5,Decreto Supremo N.° 006-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,prorroga,otro
6,Decreto Supremo N.° 005-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...,declara,otro
7,Decreto Supremo N.° 002-2022-PCM,Decreto Supremo que modifica el numeral 8.1 de...,declara,otro
8,Decreto Supremo N.° 016-2022-PCM,Decreto Supremo que declara Estado de Emergenc...,declara,otro
9,Decreto Supremo N.° 015-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...,declara,otro
11,Decreto Supremo N.° 013-2022-PCM,Prórroga del Estado de Emergencia en distritos...,prorroga,otro


#Interpretacion:
Se identificaron 22 normas con al menos una clasificación «otro». Esto sucede principalmente porque sus títulos no contienen las expresiones específicas utilizadas para identificar el motivo de la emergencia.

#11. Selección de normas relacionadas con lluvias

In [20]:
df_lluvias = df_decretos[
    df_decretos["es_lluvia"] == True
].copy()

print(
    "Total de normas relacionadas con lluvias:",
    len(df_lluvias)
)

display(
    df_lluvias[
        ["numero", "fecha", "titulo_completo", "tipo", "motivo"]
    ].head(5)
)

Total de normas relacionadas con lluvias: 1


,numero,fecha,titulo_completo,tipo,motivo
21,Decreto Supremo N.° 032-2022-PCM,1 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños


#12. Identificación de departamentos

In [21]:
import re

departamentos = [
    "Amazonas", "Áncash", "Apurímac", "Arequipa",
    "Ayacucho", "Cajamarca", "Callao", "Cusco",
    "Huancavelica", "Huánuco", "Ica", "Junín",
    "La Libertad", "Lambayeque", "Lima", "Loreto",
    "Madre de Dios", "Moquegua", "Pasco", "Piura",
    "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali"
]

def identificar_departamentos(titulo):
    encontrados = []

    for departamento in departamentos:
        patron = r"\b" + re.escape(departamento) + r"\b"

        if re.search(patron, titulo):
            encontrados.append(departamento)

    return encontrados


df_lluvias["departamentos"] = df_lluvias[
    "titulo_completo"
].apply(identificar_departamentos)

display(
    df_lluvias[
        ["numero", "departamentos"]
    ]
)

,numero,departamentos
21,Decreto Supremo N.° 032-2022-PCM,"[Amazonas, Ayacucho, Piura]"


Identificación de departamentos

Para evitar coincidencias incorrectas, se utilizaron expresiones regulares que identifican los nombres completos de los departamentos y respetan las mayúsculas.

#13. Exportación de resultados

In [22]:
from pathlib import Path

Path("datos").mkdir(exist_ok=True)

df_exportar = df_lluvias[
    ["numero", "fecha", "titulo_completo", "tipo", "motivo", "enlace"]
].copy()

df_exportar = df_exportar.rename(
    columns={"titulo_completo": "titulo"}
)

df_exportar.to_csv(
    "datos/decretos_lluvias.csv",
    index=False,
    encoding="utf-8-sig"
)

filas = []

for _, norma in df_lluvias.iterrows():
    for departamento in norma["departamentos"]:
        filas.append({
            "departamento": departamento,
            "tipo": norma["tipo"]
        })

df_departamentos = pd.DataFrame(
    filas,
    columns=["departamento", "tipo"]
)

df_resumen = pd.DataFrame({
    "departamento": departamentos
})

for tipo, columna in [
    ("declara", "declaratorias"),
    ("prorroga", "prorrogas")
]:
    conteo = (
        df_departamentos[df_departamentos["tipo"] == tipo]
        .groupby("departamento")
        .size()
    )

    df_resumen[columna] = (
        df_resumen["departamento"]
        .map(conteo)
        .fillna(0)
        .astype(int)
    )

df_resumen.to_csv(
    "datos/decretos_por_departamento.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_exportar)
display(df_resumen)

,numero,fecha,titulo,tipo,motivo,enlace
21,Decreto Supremo N.° 032-2022-PCM,1 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-lega...


,departamento,declaratorias,prorrogas
0,Amazonas,1,0
1,Áncash,0,0
2,Apurímac,0,0
3,Arequipa,0,0
4,Ayacucho,1,0
5,Cajamarca,0,0
6,Callao,0,0
7,Cusco,0,0
8,Huancavelica,0,0
9,Huánuco,0,0


In [23]:
print("Normas de lluvias:", len(df_exportar))
print("Departamentos:", len(df_resumen))

print(
    "Total de declaratorias:",
    df_resumen["declaratorias"].sum()
)

print(
    "Total de prórrogas:",
    df_resumen["prorrogas"].sum()
)

print("Archivos generados:")

for archivo in Path("datos").glob("*.csv"):
    print(archivo)

Normas de lluvias: 1
Departamentos: 25
Total de declaratorias: 3
Total de prórrogas: 0
Archivos generados:
datos\decretos_lluvias.csv
datos\decretos_por_departamento.csv
